<a href="https://colab.research.google.com/github/simichakravarty16/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026-10-14%20%E2%80%94%20Mini%20ETL%20on%20Walmart%20Sample%20%E2%80%94%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Mini-ETL on Walmart Sample

**Studio — 2026-10-14 · Fall 2026**  
**Class time:** 45 minutes
**Name:** Simi Chakravarty
---

## The midterm, in miniature

Twelve rows of the real thing. Every problem in this sample is in the full dataset you get Monday, just at a scale where you can still see the answer.

You are writing the whole pipeline as **functions**, not as loose cells. That is not stylistic — Monday you will run the same logic on a much larger file, and a pipeline built from functions is one you can re-run and hand to a teammate. A pipeline built from cells in a specific click order is not.

The extract stage is worked below as an example. Transform and load are yours to build.

In [1]:
import pandas as pd, sqlite3
from io import StringIO

RAW = '''transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2,$4.11
TXN2,fl-105,09/06/2024 10:40,PT-12,pop-tarts strawberry,breakfast,1,3.86
TXN3,FL-239,2024-09-06T11:00:00,WATER-24,Bottled Water,Grocery,-1,5.00
TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2,$4.11
TXN4,FL 239,09/06/2024 11:30,SKU#459812,Strawberry Pop-Tart,Breakfast,3,$4.11
TXN5,fl-239,2024-09-06T12:15:00,WATER-24,bottled water,grocery,4,$5.00
TXN6,FL-105,2024-09-07 08:02:00,BTL-WATER-24,Bottled Water 24pk,Grocery,6,5.00
TXN7,FL-105,2024-09-07T09:45:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,5,4.11
TXN8,FL-239,,PT-12,pop-tarts strawberry,Breakfast,2,$4.11
TXN9,FL-105,2024-09-07 14:20:00,FLASH-AA,Flashlight AA,Hardware,NULL,12.99
TXN10,FL-239,2024-09-07T15:00:00,POPTART-BLUE,Pop-Tarts Blueberry,Breakfast,1,4.11
TXN11,FL-105,09/07/2024 16:10,SKU#459812,Strawberry Pop-Tart,breakfast,4,4.11'''

### Extract — worked example

The extract function does one job: get the raw data in, and report what arrived. It does **not** clean anything. Keeping the stages separate is what lets you answer "was this a source problem or did I do it?" later.

In [2]:
def extract():
    """Read the raw transactions. No cleaning here -- report only."""
    df = pd.read_csv(StringIO(RAW))
    print(f'extracted {len(df)} rows, {len(df.columns)} columns')
    print('dtypes that arrived as text:',
          [c for c in df.columns if df[c].dtype == object])
    print('nulls:', df.isnull().sum().sum())
    print('exact duplicates:', df.duplicated().sum())
    return df

raw = extract()
raw.head()

extracted 12 rows, 8 columns
dtypes that arrived as text: ['transaction_id', 'store_id', 'timestamp', 'sku', 'product_name', 'category', 'unit_price']
nulls: 2
exact duplicates: 1


,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
0,TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2.0,$4.11
1,TXN2,fl-105,09/06/2024 10:40,PT-12,pop-tarts strawberry,breakfast,1.0,3.86
2,TXN3,FL-239,2024-09-06T11:00:00,WATER-24,Bottled Water,Grocery,-1.0,5.00
3,TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2.0,$4.11
4,TXN4,FL 239,09/06/2024 11:30,SKU#459812,Strawberry Pop-Tart,Breakfast,3.0,$4.11


Note what the report already told us: `quantity` and `unit_price` both arrived as text, there is a null, and there is a duplicate. That is your transform to-do list, generated from the data instead of from guessing.

**TODO:** before writing any transform, list the specific problems you can see in these twelve rows. There are at least six.

1. _..._
2. _..._
3. _..._
4. _..._
5. _..._
6. _..._

### Build 1 — Transform: types and duplicates

**TODO:** write `transform_types(df)` that returns a new frame with:

- exact duplicate rows removed
- `unit_price` as a float
- `quantity` numeric, with a stated decision about the null and the negative
- `store_id` normalized — note the four spellings of the same two stores
- `timestamp` as a real datetime, coercing the empty one

It must not modify the frame passed in. Print a before/after row count.

In [3]:
def transform_types(df):
    out = df.copy()

    before = len(out)
    out = out.drop_duplicates().copy()

    out["unit_price"] = pd.to_numeric(
        out["unit_price"].astype(str).str.replace("$", "", regex=False).str.strip(),
        errors="coerce"
    ).astype(float)

    # Decision: preserve missing quantities as NaN; preserve negative quantities because they may represent returns or corrections.
    out["quantity"] = pd.to_numeric(out["quantity"], errors="coerce")

    out["store_id"] = (
        out["store_id"]
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"[\s-]+", "-", regex=True)
    )

    out["timestamp"] = pd.to_datetime(
        out["timestamp"],
        errors="coerce",
        format="mixed"
    )

    print(f"Rows before deduplication: {before}")
    print(f"Rows after deduplication: {len(out)}")
    return out

typed = transform_types(raw)
print('rows:', len(raw), '->', len(typed))
typed.dtypes

Rows before deduplication: 12
Rows after deduplication: 11
rows: 12 -> 11


,0
transaction_id,object
store_id,string[python]
timestamp,datetime64[ns]
sku,object
product_name,object
category,object
quantity,float64
unit_price,float64


### Build 2 — Transform: consolidate the products

Four SKUs are strawberry Pop-Tarts. One is blueberry — leave it alone, and be ready to say why. Two are the same bottled water.

**TODO:** write `consolidate(df)` that adds a `canonical_sku` column. Any SKU not in your mapping should keep its original value, not become `NaN`.

Then print units per canonical SKU and confirm the total quantity did not change — consolidation renames things, it never adds or removes units.

In [4]:
typed

,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
0,TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2.0,4.11
1,TXN2,FL-105,2024-09-06 10:40:00,PT-12,pop-tarts strawberry,breakfast,1.0,3.86
2,TXN3,FL-239,2024-09-06 11:00:00,WATER-24,Bottled Water,Grocery,-1.0,5.00
4,TXN4,FL-239,2024-09-06 11:30:00,SKU#459812,Strawberry Pop-Tart,Breakfast,3.0,4.11
5,TXN5,FL-239,2024-09-06 12:15:00,WATER-24,bottled water,grocery,4.0,5.00
6,TXN6,FL-105,2024-09-07 08:02:00,BTL-WATER-24,Bottled Water 24pk,Grocery,6.0,5.00
7,TXN7,FL-105,2024-09-07 09:45:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,5.0,4.11
8,TXN8,FL-239,NaT,PT-12,pop-tarts strawberry,Breakfast,2.0,4.11
9,TXN9,FL-105,2024-09-07 14:20:00,FLASH-AA,Flashlight AA,Hardware,NaN,12.99
10,TXN10,FL-239,2024-09-07 15:00:00,POPTART-BLUE,Pop-Tarts Blueberry,Breakfast,1.0,4.11


In [5]:
CANON = {
    # TODO: map the strawberry Pop-Tart SKUs to one value
    "POPTART-STRAW": "POPTART-STRAW",
    "PT-12": "POPTART-STRAW",
    "SKU#459812": "POPTART-STRAW",
    # TODO: map the two bottled water SKUs to one value
    "WATER-24": "WATER-24",
    "BTL-WATER-24": "WATER-24"
}

def consolidate(df):
    out = df.copy()
    out['canonical_sku'] = (out['sku'].map(CANON).fillna(out['sku']))
    return out

consolidated = consolidate(typed)
before_total = typed["quantity"].sum()
after_total = consolidated["quantity"].sum()

print("Total quantity before:", before_total)
print("Total quantity after: ", after_total)

units_per_sku = consolidated.groupby(
    "canonical_sku", dropna=False
)["quantity"].sum()

print("\nUnits per canonical SKU:")
print(units_per_sku)


Total quantity before: 27.0
Total quantity after:  27.0

Units per canonical SKU:
canonical_sku
FLASH-AA          0.0
POPTART-BLUE      1.0
POPTART-STRAW    17.0
WATER-24          9.0
Name: quantity, dtype: float64


**Why blueberry stays separate:** _..._

### Build 3 — Transform: derive what the analysis needs

**TODO:** add `revenue` (quantity times unit price) and a `date` column from the timestamp. The `date` column is what you will join weather onto next week, so it needs to be a date — not a full timestamp, not a string.

In [6]:
def derive(df):
    out = df.copy()
    out['revenue'] = out['quantity'] * out['unit_price']
    out['date'] = pd.to_datetime(out['timestamp']).dt.date
    return out

enriched = derive(consolidated)
enriched[['transaction_id', 'canonical_sku', 'quantity', 'revenue', 'date']]

,transaction_id,canonical_sku,quantity,revenue,date
0,TXN1,POPTART-STRAW,2.0,8.22,2024-09-06
1,TXN2,POPTART-STRAW,1.0,3.86,2024-09-06
2,TXN3,WATER-24,-1.0,-5.00,2024-09-06
4,TXN4,POPTART-STRAW,3.0,12.33,2024-09-06
5,TXN5,WATER-24,4.0,20.00,2024-09-06
6,TXN6,WATER-24,6.0,30.00,2024-09-07
7,TXN7,POPTART-STRAW,5.0,20.55,2024-09-07
8,TXN8,POPTART-STRAW,2.0,8.22,NaT
9,TXN9,FLASH-AA,NaN,NaN,2024-09-07
10,TXN10,POPTART-BLUE,1.0,4.11,2024-09-07


### Build 4 — Load, and verify the load

**TODO:** write `load(df, conn)` that writes the clean frame to a `clean_sales` table, then reads back the row count and asserts it matches. Writing without verifying is how you discover at 11pm that `if_exists='append'` doubled your table.

In [7]:
def load(df, conn, table='clean_sales'):
    df.to_sql(table, conn, if_exists='replace', index=False)
    cursor = conn.cursor()
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    row_count = cursor.fetchone()[0]
    assert row_count == len(df), (
        f"Row count mismatch: DataFrame has {len(df)} rows, "
        f"but table '{table}' has {row_count} rows"
    )
    print(
        f"Load verified: {row_count} rows written to '{table}'."
    )
    pass

conn = sqlite3.connect(':memory:')
load(enriched, conn)

Load verified: 11 rows written to 'clean_sales'.


### Build 5 — run the whole thing as one pipeline

**TODO:** wire the stages together in one function, then run it twice against a fresh connection and confirm you get identical results. That is the idempotency property from Monday, tested rather than assumed.

In [8]:
def run_pipeline(conn):
    raw = extract()
    typed = transform_types(raw)
    consolidated = consolidate(typed)
    enriched = derive(consolidated)
    load(enriched, conn)
    return enriched

first = run_pipeline(sqlite3.connect(':memory:'))
second = run_pipeline(sqlite3.connect(':memory:'))

assert first.shape == second.shape, (
    f"Shape mismatch: {first.shape} != {second.shape}"
)

revenue_col = 'revenue'

first_revenue = first[revenue_col].sum(min_count=1)
second_revenue = second[revenue_col].sum(min_count=1)

assert (
    (pd.isna(first_revenue) and pd.isna(second_revenue))
    or first_revenue == second_revenue
), (
    f"Revenue mismatch: {first_revenue} != {second_revenue}"
)

print("Pipeline idempotency checks passed.")
print("First run shape:", first.shape)
print("Second run shape:", second.shape)
print("First run revenue:", first_revenue)
print("Second run revenue:", second_revenue)

extracted 12 rows, 8 columns
dtypes that arrived as text: ['transaction_id', 'store_id', 'timestamp', 'sku', 'product_name', 'category', 'unit_price']
nulls: 2
exact duplicates: 1
Rows before deduplication: 12
Rows after deduplication: 11
Load verified: 11 rows written to 'clean_sales'.
extracted 12 rows, 8 columns
dtypes that arrived as text: ['transaction_id', 'store_id', 'timestamp', 'sku', 'product_name', 'category', 'unit_price']
nulls: 2
exact duplicates: 1
Rows before deduplication: 12
Rows after deduplication: 11
Load verified: 11 rows written to 'clean_sales'.
Pipeline idempotency checks passed.
First run shape: (11, 11)
Second run shape: (11, 11)
First run revenue: 118.73
Second run revenue: 118.73


### Build 6 — the question the pipeline exists to answer

**TODO:** did strawberry Pop-Tart sales rise from Sept 6 to Sept 7? Print units by date for your canonical strawberry SKU, and say whether two days is enough evidence to conclude anything.

That second part matters. On the midterm you will have a real baseline to compare against; here you do not, and saying so is the correct answer.

In [9]:
strawberry = enriched[
    enriched['canonical_sku'] == "POPTART-STRAW"
]

units_by_date = strawberry.groupby("date")["quantity"].sum().sort_index()

print("Units by date:")
print(units_by_date)

Units by date:
date
2024-09-06    6.0
2024-09-07    9.0
Name: quantity, dtype: float64


---

## Checkpoint (participation)

Report your row count after transform, your canonical SKU count, and whether running the pipeline twice gave the same revenue total.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [10]:
# Checkpoint
rows_after_transform = 11   # TODO
canonical_skus = 4         # TODO: how many distinct products you ended with
idempotent = True             # TODO: True/False from Build 5
blueberry_call = 'We keep blueberry separate because it is a different item than strawberry poptarts, so we would not want to combine them into a canonical sku.'       # TODO: why you kept blueberry separate

print('rows after transform:', rows_after_transform)
print('canonical SKUs:', canonical_skus)
print('same result twice:', idempotent)
print('blueberry:', blueberry_call)

rows after transform: 11
canonical SKUs: 4
same result twice: True
blueberry: We keep blueberry separate because it is a different item than strawberry poptarts, so we would not want to combine them into a canonical sku.
